Leave one out reliability method

In [ ]:
%load_ext autoreload
from JR_test_scripts.escape.functions.escape_utils import load, load_homing, firing_by_bin_median_numba
from JR_test_scripts.escape.functions.escape_data_loading_funcs import extract_homing_and_escape_periods
from JR_test_scripts.escape.functions.escape_tuning_funcs import tuning_method
from behave_analysis.utils.creating_directories import make_directory

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from behave_analysis.database.Experiments.JAL006_ex import JAL6_flip4_21mar
exp = JAL6_flip4_21mar

%autoreload 2
comp = 'escape'
# load data
session, frame_by_cluster_matrix, behave, y_pos, x_pos, bar, barflip, escape, outofshelter = load(exp)
ons, offs, homie = load_homing(session, len(behave))

2025-02-03 16:12:21.379 | INFO     | behave_analysis.utils.data_loading:load_or_extract_homings:35 - Homings object found. Loading...


In [3]:
def leave_one_out_reliability(var, escape_matrix, cond, h_start, bins, n_cond, n_neur):
    """Computes for each cell in each condition the average correlation coefficient"""
    
    # initialize variables for output
    reliability = np.full((n_cond, n_neur), np.nan)
    fr_full = np.zeros((n_cond, n_neur, bins)) # conditions x neurons x n_bins
    c = [len([x for x in h_start if cond[x] == i]) for i in range(n_cond)]
    mat_num_cond = np.full((n_cond, n_neur, max(c),bins), np.nan) # conditions x neurons x trials x bins

    # iterate through conditions
    for i in range(n_cond):
        i = int(i)
        # start by condition
        cond_start = [x for x in h_start if cond[x] == i]
        cond_start.append(np.sum(cond == i))
        # iterate through neurons
        for j, n in enumerate(escape_matrix):
            # iterate through trials, pull out firing by bin
            for tr, _ in enumerate(cond_start[:-1]):
                neur = n[cond_start[tr]:cond_start[tr+1]]
                v = var[cond_start[tr]:cond_start[tr+1]]
                mat_num_cond[i, j, tr,:] = firing_by_bin_median_numba(v.astype(int), neur, bins, remove_empty = False)
            
            mat = mat_num_cond[i, j, :,:]
            # step 4: take median across trials
            all_nan_cols = np.all(np.isnan(mat), axis=0)
            smoothed_firing_rates = np.full(bins, np.nan)
            smoothed_firing_rates[~all_nan_cols] = np.nanmedian(mat[:,~all_nan_cols], axis = 0)
            # dump together for output
            fr_full[i, j, :] = smoothed_firing_rates

            # leave one out median
            tr_corr_coeff = np.full(mat_num_cond.shape[2], np.nan)
            tr_rms = np.full(mat_num_cond.shape[2], np.nan)
            if np.sum(smoothed_firing_rates) > 0:
                for tr in range(mat_num_cond.shape[2]):
                    loo_mat = np.delete(mat, tr, axis = 0)
                    all_nan_cols = np.all(np.isnan(loo_mat), axis=0)
                    loo = np.full(bins, np.nan)
                    loo[~all_nan_cols] = np.nanmedian(loo_mat[:,~all_nan_cols], axis = 0)
                    # corr coeff for each trial
                    tr_corr_coeff[tr] = np.corrcoef(loo, mat[tr,:])[0, 1]
                    tr_rms[tr] = (np.sqrt(np.mean(mat[tr,:]**2)))
                # average across trial
                id_nans = np.logical_or(np.isnan(tr_corr_coeff), np.isnan(tr_rms))
                reliability[i,j] = np.average(tr_corr_coeff[~id_nans], weights = tr_rms[~id_nans])
    
    return fr_full, mat_num_cond, reliability

In [ ]:
"""Compute leave one out reliability for ech cell in each condition and then plot it"""

c_names = ['shelter_only', 'barrier', 'barrier_flipped']
colors = ['#228B22','#FF8C00','#008B8B']

fcm = gaussian_filter1d(frame_by_cluster_matrix, 2, axis = 0)
n_neur = frame_by_cluster_matrix.shape[1]

for comp in ['escape']:#['escape','bird_dist_shelter']:

    # the real stat
    var, escape_matrix, cond, h_start = extract_homing_and_escape_periods(session, 
                                                                            fcm, 
                                                                            behave, 
                                                                            y_pos, x_pos, 
                                                                            bar, 
                                                                            barflip, 
                                                                            comp, 
                                                                            ons, offs, 
                                                                            no_stationary = False, 
                                                                            return_escape = False,
                                                                            zscore = False)

    # initialize vars
    bins = int(np.amax(var)+1)
    n_cond = len(np.unique(cond))

    fr_full, mat_num_cond, reliability = leave_one_out_reliability(var, escape_matrix, cond, h_start, bins, n_cond, n_neur)

    """Plotting"""
    nickname = exp.nick_name + '_' + exp.experiment_date
    exp_nickname = exp.nick_name + '_' + exp.experiment_date + '_' + comp
    dump_path = make_directory("Z:/Jasmine_Laurence/summary_plots/tuning_curves/" + nickname + "/" + exp_nickname)

    for neur in np.arange(fr_full.shape[1]):
        fig, axs = plt.subplots(1,3, figsize = (12,4), constrained_layout=True)

        # compute min/max for this neuron across all conditions
        vmin, vmax = np.nanmin(mat_num_cond[:,neur,:,:]), np.nanmax(mat_num_cond[:,neur,:,:])  # Ignore NaNs

        # compute min/max for the average also
        ylim = [np.nanmin(fr_full[:,neur,:]), np.nanmax(fr_full[:,neur,:])]

        for c in [0,1,2]:
            nan_rows = np.all(np.isnan(mat_num_cond[c,neur,:,:]), axis=1)
            im = axs[c].imshow(mat_num_cond[c,neur,~nan_rows,:], cmap="gray_r", vmin = vmin, vmax = vmax, aspect="auto", interpolation = "none")
            axs[c].set_title(c_names[c] + f'\n Reliability = {reliability[c,neur]:.2f}')
            axs[c]. set_xlabel(comp)
            if c == 0:
                axs[c].set_ylabel('trials')

            ax2 = axs[c].twinx()
            ax2.plot(fr_full[c,neur,:], linewidth = 2, color = colors[c])
            ax2.spines["right"].set_color(colors[c])
            ax2.tick_params(axis="y", colors=colors[c])  # Change tick color
            ax2.yaxis.label.set_color(colors[c])  # Change axis label color
            ax2.set_ylim(ylim)
            
            if c == 2:
                ax2.set_ylabel('Firing rate')
                cbar = fig.colorbar(im, ax=axs[c], location="right", pad=0.1)
                cbar.set_label('Firing rate')

            fig.savefig(dump_path + "/neuron" + str(neur) + "_loo_reliability.png")
            plt.close()

c:\Users\Jasmine\miniconda3\envs\JAL2pipeline\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Jasmine\miniconda3\envs\JAL2pipeline\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\Jasmine\miniconda3\envs\JAL2pipeline\lib\site-packages\numpy\lib\_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Jasmine\miniconda3\envs\JAL2pipeline\lib\site-packages\numpy\lib\_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
